# 03 · Model de riscos proporcionals de Cox

$$h(t \mid x) = h_0(t)\,\exp(\beta_1 x_1 + \dots + \beta_p x_p)$$

Cada *hazard ratio* $HR = e^{\beta}$ indica quantes vegades es multiplica el risc instantani de
baixa quan la variable passa de la categoria de referència a la indicada, **mantenint la resta
constants**. El model s'ajusta amb el conjunt *train* i s'avalua amb el *test*.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.precision", 3)
from src.preparacio import carregar_dades, codificar_cox, dividir_train_test
from src.cox import (ajustar_cox, taula_hazard_ratios, comprovar_ph, c_index, preparar_estratificat,
                     comparar_parametrics, NOMS_COVARIABLES)
from src.grafics import fig_forest, exportar
df = carregar_dades()
tr, te = dividir_train_test(df)
X_tr, X_te = codificar_cox(tr), codificar_cox(te)
cph = ajustar_cox(X_tr)

## Hazard ratios

In [2]:
hr = taula_hazard_ratios(cph)
fig = fig_forest(hr, "Hazard ratios del model de Cox (IC 95 %)", NOMS_COVARIABLES)
exportar(fig, "cox_forest"); fig.show()
hr.assign(variable=hr.variable.map(NOMS_COVARIABLES))

,variable,HR,IC_inf,IC_sup,p_valor
0,Contracte 2 anys (vs mensual),0.073,0.056,0.094,3.661e-88
1,Contracte 1 any (vs mensual),0.208,0.174,0.248,8.852e-67
2,Sense internet (vs DSL),0.338,0.265,0.432,3.294e-18
3,Transferència automàtica (vs xec postal),0.517,0.431,0.621,1.535e-12
4,Targeta automàtica (vs xec postal),0.527,0.438,0.634,1.215e-11
5,Té parella,0.562,0.501,0.631,1.885e-22
6,Seguretat en línia,0.567,0.492,0.654,5.168e-15
7,Suport tècnic,0.686,0.596,0.789,1.210e-07
8,Té persones a càrrec,0.882,0.763,1.019,8.880e-02
9,Client sènior,0.917,0.812,1.035,1.596e-01


**Lectura.** Un contracte de 2 anys redueix el risc instantani de baixa a menys d'una
desena part del d'un contracte mensual; la fibra òptica el duplica respecte a DSL. La quota
mensual apareix lleugerament protectora *un cop ajustada* pels serveis contractats: l'efecte
"car → marxa" que es veu a Kaplan-Meier queda absorbit sobretot per la fibra (variables
correlacionades).

## Capacitat predictiva: índex de concordança (C-index)

Probabilitat que, d'un parell de clients comparables, el model assigni més risc al que marxa
abans. 0,5 = atzar; 1 = ordenació perfecta. És l'equivalent de l'AUC per a dades censurades.

In [3]:
pd.DataFrame({"conjunt": ["train", "test"],
              "C-index": [c_index(cph, X_tr), c_index(cph, X_te)]})

,conjunt,C-index
0,train,0.859
1,test,0.839


## Comprovació del supòsit de riscos proporcionals (residus de Schoenfeld)

El model suposa que cada HR és constant en el temps. Si $p < 0{,}05$ el supòsit es viola.

In [4]:
ph = comprovar_ph(cph, X_tr)
ph.assign(variable=ph.variable.map(NOMS_COVARIABLES))

,variable,p_valor,viola
0,Quota mensual (+1 €),1.554e-08,True
1,Contracte 1 any (vs mensual),1.445e-06,True
2,Targeta automàtica (vs xec postal),3.715e-05,True
3,Suport tècnic,5.425e-04,True
4,Transferència automàtica (vs xec postal),1.153e-03,True
5,Té parella,3.628e-02,True
6,Seguretat en línia,4.144e-02,True
7,Xec electrònic (vs xec postal),5.261e-02,False
8,Fibra òptica (vs DSL),9.831e-02,False
9,Sense internet (vs DSL),1.083e-01,False


Diverses variables violen el supòsit; la més forta és la **quota mensual**, seguida del
contracte d'1 any. És esperable: l'efecte protector d'un contracte no és el mateix el primer mes
que al final del compromís. Per al contracte, la solució habitual és **estratificar**: permetre
una funció de risc base $h_0(t)$ diferent per a cada tipus de contracte i estimar la resta
d'efectes conjuntament. Això **només corregeix la violació del contracte**.

In [5]:
X_tr_e = preparar_estratificat(tr)
cph_e = ajustar_cox(X_tr_e, strata=["Contract"])
hr_e = taula_hazard_ratios(cph_e)
fig = fig_forest(hr_e, "Hazard ratios — Cox estratificat per contracte", NOMS_COVARIABLES)
exportar(fig, "cox_forest_estratificat"); fig.show()
X_te_e = preparar_estratificat(te)
print(f"C-index test (estratificat): {c_index(cph_e, X_te_e):.3f}")

C-index test (estratificat): 0.798


Supòsit de riscos proporcionals al model estratificat:

In [6]:
ph_e = comprovar_ph(cph_e, X_tr_e)
ph_e.assign(variable=ph_e.variable.map(NOMS_COVARIABLES))

,variable,p_valor,viola
0,Quota mensual (+1 €),1.805e-09,True
1,Targeta automàtica (vs xec postal),3.355e-04,True
2,Suport tècnic,8.160e-04,True
3,Transferència automàtica (vs xec postal),6.167e-03,True
4,Fibra òptica (vs DSL),1.356e-02,True
5,Té parella,4.049e-02,True
6,Seguretat en línia,8.181e-02,False
7,Sense internet (vs DSL),1.384e-01,False
8,Xec electrònic (vs xec postal),1.742e-01,False
9,Té persones a càrrec,7.523e-01,False


La quota mensual, el pagament automàtic, el suport tècnic i altres variables continuen
violant el supòsit: tenen **efectes que canvien amb el temps**. Es podrien modelar amb
interaccions amb el temps (model de Cox amb covariables dependents del temps), però queda fora
de l'abast d'aquest cas pràctic i es documenta com a limitació.

## Models paramètrics (comparació breu)

Ajust univariant de la durada amb distribucions Weibull i exponencial (menor AIC = millor).

In [7]:
comparar_parametrics(df)

,model,AIC
0,Weibull,21156.573
1,Exponencial,21696.972


El Weibull ajusta clarament millor que l'exponencial: el risc **no és constant** en el temps
(és molt alt els primers mesos), cosa que confirma l'anàlisi del risc temporal (notebook 04).

## Conclusions

- **C-index ≈ 0,84 en test**: el model ordena bé els clients per risc, sense sobreajust
  apreciable (train ≈ 0,86).
- **Factors de risc**: fibra òptica (HR ≈ 2,2), factura electrònica. **Factors protectors**:
  contractes d'1 i 2 anys (HR ≈ 0,21 i 0,07), pagament automàtic, seguretat en línia, suport
  tècnic, tenir parella.
- El supòsit de riscos proporcionals **no es compleix per a diverses variables**. Estratificar pel
  contracte en corregeix la violació i manté HR similars per a la resta, però la quota mensual i
  altres variables tenen efectes que varien amb el temps: els seus HR s'han de llegir com un
  efecte mitjà al llarg del període.
- El C-index de l'estratificat (≈ 0,80) no és directament comparable: el contracte ja no forma
  part del predictor lineal sinó de la funció de risc base, i el C-index només avalua el predictor.